# Semi-Supervised Securitisation Classifier - Threat-Only Auxiliary

In this last notebook I implement my threat-only auxiliary variant of the multi-head securitisation regression model. It extends my baseline by adding a single auxiliary regression head trained on just the threat lexicon's pseudo-labels over the ~44k unlabelled corpus sentences. The auxiliary head is discarded at inference, so the model's output signature is identical to the baseline. The architecture, hyperparameters, training configuration, evaluation procedure and outputs are all kept identical to my baseline. I created this variant as a controlled comparison against the full auxiliary variant. Here I restrict the supervision signal to the strongest lexicon-gold alignment (threat) to test if focusing the gradient on just the cleanest signal avoids the noise that the weaker urgency and justification lexicons might amplify. 

For the full description of this auxiliary supervision mechanism see Section 3.4-3.6 of my paper.

In [1]:
# I set all my imports at the top for cleanliness's sake
import copy
import random
import re
import itertools
import warnings
from difflib import SequenceMatcher
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, cohen_kappa_score
from scipy.stats import pearsonr, spearmanr
from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm

In [ ]:
# I set up this dict where every hyperparameter goes. I did this so the config is easy to scan & compare against my other variants
# Everything above the semi-supervised block is identical to my baseline. Only the keys in the semi-supervised block are new
threat_only_aux_hyperparams= {
    "MODEL_NAME": "microsoft/deberta-v3-base",
    "MAX_LEN": 256,
    "RANDOM_SEED": 42,
    "TRAINING_EPOCHS": 15,
    "EARLY_STOPPING_PATIENCE": 5, # this stops the training if val loss doesn't improve for this many epochs. Idea is it protects against overfitting, which is needed on my small gold set
    "PHYSICAL_BATCH_SIZE": 8, # this is the actual per-step batch. I kept it small to fit in the 10GB GPU memory I get on NCC
    "GRAD_ACCUM_STEPS": 2, # this accumulating 2 steps restores effective batch size of 16
    "BASE_LR": 2e-5,
    "LLRD_FACTOR": 0.9, # layer-wise LR decay factor per Sun et al. [8]. Here lower layers get littler updates
    "WARMUP_PROPORTION": 0.1, # this determines proportion of total steps in linear warmup before cosine decay [9]
    "GRAD_CLIP_MAX_NORM": 1.0, # from [13] (Pascanu et al.), its standard for transformer fine-tuning
    "HEAD_DROPOUT": 0.3, # this strong-ish dropout addresses head-level overfitting on my small gold training set
    "NUM_DROPOUT_SAMPLES": 2, # I have multi-sample dropout per Inoue [4], K=2 gives most of the benefit. I tried bigger K but it crashed on the limited NCC GPU
    "LAMBDA_RDROP": 0.1, # my R-Drop consistency weight [12]
    "LOSS_WEIGHT_THREAT": 1.0,
    "LOSS_WEIGHT_URGENCY": 1.75, # here I've upweighted it to offset its weaker lexical signal & lower high-intensity freq. in my gold labels
    "LOSS_WEIGHT_JUSTIFICATION": 1.75, # & I've upweighted for same reason as urgency
    "LABELLED_DATA_FILENAME": "manifesto_corpus_annotation_batches.xlsx",
    "PRE2005_AUG_FILENAME": "pre2005_annotation_workbook.xlsx",

    # these are the specific hyperparams I added over the baseline, and are the only ones that vary
    "WEAK_LABELS_FILENAME": "manifesto_corpus_weak_labels_scored.xlsx",
    "LAMBDA_AUX_WEAK": 0.50, # this is the weight on the auxiliary loss for the weak pool. It's the primary new hyperparam & it controls how much the ~44k weak pool sentences influence the shared encoder through the aux heads.
    "WEAK_BATCHES_PER_GOLD_BATCH": 6, # 6:1 weak-to-gold batch ratio. i.e. per gold batch I process 6 weak batches through the aux pathway
    "WEAK_POOL_SUBSAMPLE_FRACTION": 1.0, # I deliberately don't LDS-weight the weak pool. The weak labels are already noisy & adding a density-based reweight on top would amplify noise on the rare high-intensity pseudo-labels
    "APPLY_LDS_TO_WEAK_POOL": False, # I deliberately don't LDS-weight the weak pool. The weak labels are already noisy & adding a density-based reweight on top would amplify noise on the rare high-intensity pseudo-labels
}

In [3]:
# I keep all the RNGs are here. I use a single seed so the split, initialisations, dropout masks & shuffling are reproducible across runs
# I also enabled cudnn determinism is enabled too to make sure I get maximum reproducibility on my model
random_seed= threat_only_aux_hyperparams["RANDOM_SEED"]
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)
torch.backends.cudnn.deterministic= True
torch.backends.cudnn.benchmark= False

# getting NCC GPU, which is really much a must for training (otherwise it takes ages)
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")
print(f"random seed fixed at {random_seed}")

using device: cuda
random seed fixed at 42


### Loading the labelled data

Now I load the hand-annotated manifesto corpus. Every sentence was labelled by me using the codebook I created for this project (see my Appendix A), with three ordinal-scale scores in [0, 3] that cover threat articulation, urgency expression and justification of extraordinary power. I've stored annotation batches as separate sheets inside that 1 excel notebook below, which I merge, validate & normalise to [0, 1] in the below code.

In [4]:
# 1st I load in the gold labels. If you want to examine them, each sheet is a (150-len) batch of my annotations
labelled_data_filename= threat_only_aux_hyperparams["LABELLED_DATA_FILENAME"]
print(f"loading labelled data from: {labelled_data_filename}")

try:
    all_sheets_by_name= pd.read_excel(labelled_data_filename, sheet_name=None)
    print(f"workbook loaded successfully, containing {len(all_sheets_by_name)} annotation batch sheets")
except Exception as load_e:
    print(f"couldn't find the file - please check it's in the working directory. error: {load_e}")

loading labelled data from: manifesto_corpus_annotation_batches.xlsx
workbook loaded successfully, containing 10 annotation batch sheets


In [5]:
# 2nd I combine every sheet that has a name starting with "Batch_" into 1 df
# I set it up this way so I could add new batch sheets to the workbook without having to mess around with this code
batch_sheets= [df for sheet_name, df in all_sheets_by_name.items() if sheet_name.startswith("Batch_")]

if batch_sheets:
    labelled_data_masterset= pd.concat(batch_sheets, ignore_index=True)
    print(f"merged {len(batch_sheets)} batch sheets into a single dataframe")
    print(f"total row count before filtering: {len(labelled_data_masterset)}")
else:
    print("no sheets found starting with 'Batch_' - is the right workbook loaded?")
    labelled_data_masterset= pd.DataFrame()

merged 10 batch sheets into a single dataframe
total row count before filtering: 1500


In [6]:
# here for a row to count as labelled, each of the 3 score cols needs a numeric value in [0, 3]
dim_score_cols= ["threat_score", "urgency_score", "justification_score"]

# here I get rid of nans (which would be there if I missed out labelling any dims for a sentence)
for dim in dim_score_cols:
    labelled_data_masterset[dim]= pd.to_numeric(labelled_data_masterset[dim], errors="coerce")
check_if_valid_row= (labelled_data_masterset[dim_score_cols].notna().all(axis=1) & labelled_data_masterset[dim_score_cols].ge(0).all(axis=1) & labelled_data_masterset[dim_score_cols].le(3).all(axis=1))
available_labelled_data= labelled_data_masterset[check_if_valid_row].copy() # here I'm only taking rows that are valid (so no nans)
rows_gotten_dropped= len(labelled_data_masterset)-len(available_labelled_data)

print(f"there are {len(available_labelled_data)} rows to train this model on.")
print(f"got rid of {rows_gotten_dropped} because of invalid/missing labels.")

available_labelled_data.head() # for examining below, should now show labels on a 0-3 scale (but most of them are 0 or one scored as securitisation is comparativley rare in manifestos)

there are 1500 rows to train this model on.
got rid of 0 because of invalid/missing labels.


,sentence_id,sentence_text,threat_score,urgency_score,justification_score
0,769,Encouraging local authorities to use their new...,0,0,0
1,162,Both these parties are for Brexit,0,0,0
2,367,This will expand community owned renewable ene...,0,0,1
3,904,Support the creation of aUK Constitutional Con...,0,0,0
4,929,Establishing a 'Fair Deal for Service Personne...,0,0,0


In [7]:
# now I normalise the [0, 3] ordinal vals to [0, 1], which keeps magnitudes controlled & matches what the model's sigmoid-bounded heads expect
for dim in dim_score_cols:
    available_labelled_data[dim]= available_labelled_data[dim] / 3.0

### Supplementary pre-2005 augmentation

I found that the gold training set was quite heavily imbalanced in urgency and justification specifically, with relatively few high-intensity examples of either of those dimensions. To partially address this, I supplemented the corpus with hand-labelled 1987-2001 manifesto sentences that were selected by running the original baseline model over that corpus and annotating them to the same codebook (my paper's Appendix A). These augmented sentences are restricted just to the training partition since their model-assisted selection would otherwise introduce bias if used in validation or test. The merge happens before the split so my split logic can correctly deal with the augmented rows.

In [8]:
# 1st I merge the pre-2005 hand-labelled new set into my current working df & flag them so the split step can ensure they're only kept in train set part
pre2005_aug_filename= threat_only_aux_hyperparams["PRE2005_AUG_FILENAME"]

# I have 2 candidate sheets in the workbook, 1 per dimension that needed augmenting
augmented_urgency_labs= pd.read_excel(pre2005_aug_filename, sheet_name="urgency_candidates", usecols=["sentence_text", "threat_label", "urgency_label", "justification_label"])
augmented_just_labs= pd.read_excel(pre2005_aug_filename, sheet_name="justification_candidates", usecols=["sentence_text", "threat_label", "urgency_label", "justification_label"])
for aug_df in [augmented_urgency_labs, augmented_just_labs]: # here I'm renaming col headres so they match the gold set
    aug_df.rename(columns={"threat_label": "threat_score", "urgency_label": "urgency_score", "justification_label": "justification_score"}, inplace=True)
augmented_urgency_labs= augmented_urgency_labs.dropna(subset=dim_score_cols) # & dropping any rows I left unannotated
augmented_just_labs= augmented_just_labs.dropna(subset=dim_score_cols)
aug_labs_combined= pd.concat([augmented_urgency_labs, augmented_just_labs], ignore_index=True) # then combining & de-duplicating across the 2 candidate sheets
aug_labs_combined= aug_labs_combined.drop_duplicates(subset="sentence_text", keep="first") # in case I had duplicates that scored both in top 150 in urgency & top 100 in justification and x2 labelled them

# now I cut any sentence that's already in the gold set so I don't double-count
existing_gold_texts= set(available_labelled_data["sentence_text"].str.strip())
aug_labs_combined= aug_labs_combined[~aug_labs_combined["sentence_text"].str.strip().isin(existing_gold_texts)].copy()
for score_col in dim_score_cols: # I normalise to [0, 1] the same way as I did for the gold labels
    aug_labs_combined[score_col]= pd.to_numeric(aug_labs_combined[score_col], errors="coerce") / 3.0
aug_labs_combined= aug_labs_combined.dropna(subset=dim_score_cols)

# this bit assigns the (non-clashing) sentence_ids
max_existing_id= available_labelled_data["sentence_id"].max() if "sentence_id" in available_labelled_data.columns else len(available_labelled_data)
aug_labs_combined["sentence_id"]= range(int(max_existing_id) + 1, int(max_existing_id) + 1 + len(aug_labs_combined))

# & highlight augmented rows so the stratified split put them in train only
available_labelled_data["_is_augmented"], aug_labs_combined["_is_augmented"]= False, True
available_labelled_data= pd.concat([available_labelled_data, aug_labs_combined], ignore_index=True) # both then combined

print(f"So, the gold set size (pre-augmentation) is {(~available_labelled_data['_is_augmented']).sum()} rows")
print(f"augmented sentences added {available_labelled_data['_is_augmented'].sum()} to that (after any deduplicating went on)")
print(f"bringing the combined pool size up to {len(available_labelled_data)} labels")
print(f"so No of urgency >= 2/3 is now {(available_labelled_data['urgency_score'] >= 2/3).sum()} labels & No of justification >= 2/3 now {(available_labelled_data['justification_score'] >= 2/3).sum()}.")

So, the gold set size (pre-augmentation) is 1500 rows
augmented sentences added 248 to that (after any deduplicating went on)
bringing the combined pool size up to 1748 labels
so No of urgency >= 2/3 is now 158 labels & No of justification >= 2/3 now 108.


### Stratified train / validation / test split

Here I do the partitioning, splitting the gold-labelled pool into 70/15/15 train, validation and test subsets & stratifying by a composite intensity key (the mean score across dimensions, binned into quartiles) so rare high-intensity sentences are proportionally spread out across partitions instead of concentrating in one by chance, following [1]'s methodology. The pre-2005 augmented sentences are only put in the training partition to keeping validation & test free from any model-assisted selection bias.

In [9]:
# note: I learnt the fundamental of implementing stratified splits in code from this repo, and adapted it to my use case here: https://github.com/trent-b/iterative-stratification
# here i'm separating gold from augmented before splitting, because stratification is computed on the gold rows & augmented rows get added back into train afterwards
pattern_to_seperate_goldonly= ~available_labelled_data["_is_augmented"]
gold_only_rows= available_labelled_data[pattern_to_seperate_goldonly].copy()
augmented_only_rows= available_labelled_data[~pattern_to_seperate_goldonly].copy()

# now for the stratification, for which I use quartile bins on the mean score across dimensions
gold_only_rows["_stratification_key"]= pd.qcut(gold_only_rows[dim_score_cols].mean(axis=1),q=4, labels=False, duplicates="drop")
train_gold, temp_df= train_test_split(gold_only_rows, test_size=0.3, random_state=random_seed, stratify=gold_only_rows["_stratification_key"])
val_df, test_df= train_test_split(temp_df, test_size=0.5, random_state=random_seed, stratify=temp_df["_stratification_key"])

train_df= pd.concat([train_gold, augmented_only_rows], ignore_index=True) # aug in training only as I described

for cleanup_each_df in [train_df, val_df, test_df, gold_only_rows]: # header cols go, not needed anymore
    cleanup_each_df.drop(columns=["_stratification_key", "_is_augmented"], errors="ignore", inplace=True)
available_labelled_data.drop(columns=["_is_augmented"], errors="ignore", inplace=True)

print(f"The model has this many labels: {len(train_df)} train (incl. {len(augmented_only_rows)} augmented labels), {len(val_df)} validation and {len(test_df)} test.")

# I do a quick check on the stratification here just to check it's all as I expected
print("the intensity distribution across gold partitions is:")
for partition_name, partition_df in [("train (gold)", train_gold), ("val", val_df), ("test", test_df)]:
    non_zero_frac= (partition_df[dim_score_cols] > 0).any(axis=1).mean()
    urgency_high= (partition_df["urgency_score"] >= 2/3).sum()
    justification_high= (partition_df["justification_score"] >= 2/3).sum()
    print(f"  {partition_name} has {non_zero_frac:.1%} non-zero. Label count w/ urgency>=2 is {urgency_high} & with justification>=2 is {justification_high}")

The model has this many labels: 1298 train (incl. 248 augmented labels), 225 validation and 225 test.
the intensity distribution across gold partitions is:
  train (gold) has 42.1% non-zero. Label count w/ urgency>=2 is 32 & with justification>=2 is 41
  val has 38.2% non-zero. Label count w/ urgency>=2 is 4 & with justification>=2 is 7
  test has 43.6% non-zero. Label count w/ urgency>=2 is 8 & with justification>=2 is 12


### Loading the weakly-labelled corpus

This is the first addition over my baseline. The weak-labels file contains the ~44k manifesto sentences from the full corpus, scored by my dimension-specific lexicons (see my paper, Section 3.3) which produce continuous pseudo-labels in [0, 1] for each of the three securitisation dimensions. These pseudo-labels are noisy proxies for the true rhetorical intensity but they extend the supervisory signal far beyond the ~1.3k gold sentences. After loading I deduplicate the weak corpus against my gold set so the auxiliary heads don't see gradient on sentences that are also gold-supervised, which would just amplify the gold loss in disguise.

In [ ]:
# here I'm loading in the weakly-labelled corpus & deduplicating against my gold set
# the weak score cols are dimension-specific pseudo-labels w/ range [0, 1] (see my paper Section 3.3 for more details on that)
weak_score_cols_all= ["threat_weak_score", "urgency_weak_score", "justification_weak_score"]
weak_corpus_df= pd.read_excel(threat_only_aux_hyperparams["WEAK_LABELS_FILENAME"], usecols=["sentence_text"] + weak_score_cols_all)
weak_corpus_df= weak_corpus_df.drop_duplicates(subset="sentence_text", keep="first")
print(f"got the full weak corpus loaded in with {len(weak_corpus_df)} sentences")

# now I deduplicate weak against gold so the aux heads don't get gradient on sentences that the gold heads are also supervised on (would just amplify gold loss in disguise)
gold_texts_list= available_labelled_data["sentence_text"].tolist()
def normalise_text_for_dedup_backup_func(sent): # I wrote this *just* in case there was any font-formatting inconsistencies amongst the labels, which there shouldn't be so it should never need to do anything functionally
    sent= str(sent).strip().lower()
    sent= re.sub(r'[^\w\s]', '', sent)
    sent= re.sub(r'\s+', ' ', sent)
    return sent
gold_normalised_set= {normalise_text_for_dedup_backup_func(sent) for sent in gold_texts_list}
weak_corpus_df["_is_gold_match"]= weak_corpus_df["sentence_text"].apply(lambda x: normalise_text_for_dedup_backup_func(x) in gold_normalised_set)
n_gold_matches_in_weak= weak_corpus_df["_is_gold_match"].sum()
print(f"the exact-match dedup found {n_gold_matches_in_weak} weak rows that overlap with gold")

weak_only_df= weak_corpus_df[~weak_corpus_df["_is_gold_match"]].copy()
weak_only_df= weak_only_df.drop(columns=["_is_gold_match"]).reset_index(drop=True)
print(f"weak pool after dedup is {len(weak_only_df)} sentences strong")

# quick sanity check of non-zero in the threat dim
print(f"threat_weak_score has {(weak_only_df['threat_weak_score'] > 0).mean()*100:.1f}% non-zero")

# this is a safeguard I put in place to drop any rows where sentence_text isn't a string (so it catches nan/float values that would crash the tokeniser later)
weak_only_df= weak_only_df[weak_only_df["sentence_text"].apply(lambda x: isinstance(x, str))].reset_index(drop=True)
print(f"weak pool after dropping any non-string rows is {len(weak_only_df)} sentences.")

got the full weak corpus loaded in with 45114 sentences
the exact-match dedup found 1468 weak rows that overlap with gold
weak pool after dedup is 43646 sentences strong
threat_weak_score has 13.5% non-zero
weak pool after dropping any non-string rows is 43644 sentences.


### Label Distribution Smoothing

My gold training set is heavily skewed toward zero-valued labels, which reflects how rare securitising language is in political manifestos. A standard SmoothL1 loss on its own would have treated every sample equally so the gradient signal would be dominated by the many zero-labelled examples I have, and the model would thus under-train on the rare high-intensity sentences that are most theoretically informative to it. To address that I apply Label Distribution Smoothing [2], which estimates the effective label density per dimension through a Gaussian-smoothed histogram & assigns each training sample a loss weight that's inversely proportional to its (local) density. This gives rare samples a larger gradient weight overall whilst keeping the gradient scale preserved with mean-1 normalisation. I deliberately don't apply LDS to the weak pool (which I control w/ APPLY_LDS_TO_WEAK_POOL in the hyperparams above) because the weak labels are already noisy and I don't want to exasperate that noise on the rare high-intensity pseudo-labels.

In [11]:
# note: [2] had a github link that I read through and drew inspiration from to tailor the idea to my context. You can find it at: https://github.com/YyzHarry/imbalanced-regression

# the problem I had was that my gold set is hugely skewed toward 0 because most political sentences just aren't securitising.
# If I trained with a plain per-sample loss the gradient would dominated by the many zero-labelled examples & the model would basically learn "predict near zero" for everything
# During my reading I came across the idea of LDS, which actually fixes my specific problem because it gives rare labels (the high-intensity sentences I actually care about most) bigger weights and common labels smaller ones
# It does this by giving each training sample a loss weight that's inversely proportional to how dense its label region is.
# The final mean-1 rescaling is there for keeping the overall loss magnitude comparable to what an unweighted SmoothL1 baseline would give, so the val/test losses I report are still interpretable, which I need for my paper.

def compute_lds_weights(training_labels_matrix, num_histogram_bins=10, gaussian_kernel_sigma=1.0):
    # training_labels_matrix is an (N, D) float array of normalised [0, 1] training labels & I return an (N,) float32 weight array w/ mean 1.0
    num_training_samples, num_dimensions= training_labels_matrix.shape
    weights_per_sample_per_dim= np.ones((num_training_samples, num_dimensions), dtype=np.float32)

    # I do this dimension by dimension because each dimension has its own label distribution to deal with (threat is more spread out, urgency & justification are heavier on zeros)
    for idx_of_dim in range(num_dimensions):
        labels_for_this_dim= training_labels_matrix[:, idx_of_dim]
        # histogram over [0, 1] is my raw density estimate for this dim. I then smooth w/ a Gaussian which pools count info across neighbouring bins, which is what [2] recommend 
        # & what stops 1 outlier sentence from being sat in an otherwise empty bin from getting a ridiculous weight (which could happen on urgency/justification)
        cnt_of_raw_bins, bin_edges= np.histogram(labels_for_this_dim, bins=num_histogram_bins, range=(0.0, 1.0))
        smoothed_density_per_bin= np.maximum(gaussian_filter1d(cnt_of_raw_bins.astype(float), sigma=gaussian_kernel_sigma), 1e-6)
        # for each individual label I look up which bin it lives in & weight inversely by the smoothed density there
        # dense bins (mostly zeros) get low weight, sparse bins (rare high-intensity sentences) get high weight which is what I need
        bin_index_per_label= np.clip(np.digitize(labels_for_this_dim, bin_edges[1:]), 0, num_histogram_bins - 1)
        weights_for_this_dim= 1.0 / smoothed_density_per_bin[bin_index_per_label]
        weights_for_this_dim /= weights_for_this_dim.mean() # this is that per-dim mean-1 rescaling I mentioned above, so no 1 dim's weight scale dominates the cross-dim avg below
        weights_per_sample_per_dim[:, idx_of_dim]= weights_for_this_dim

    # now I collapse the (N, D) matrix down to 1 weight per sample by averaging across dims
    # I had to make a decision on the code here, as max would emphasise the rarest dim but mean is gentler
    # I ended up choosing mean because a sample that's rare in any 1 dim still ends up upweighted overall & it keeps the cross-dim contribution balanced
    final_per_sample_weights= weights_per_sample_per_dim.mean(axis=1)
    final_per_sample_weights /= final_per_sample_weights.mean() # 1 last mean-1 rescaling, which is what makes my LDS-weighted loss directly comparable in magnitude to an unweighted SmoothL1 run so the loss values I print during training stay interpretable
    return final_per_sample_weights.astype(np.float32)


# only the training split gets LDS-weighted here. I'm deliberately keeping val & test uniform so the losses I report on those splits can be compared directly to a standard SmoothL1 baseline & don't mix the weighting scheme into my evaluation
training_labels_for_lds= train_df[dim_score_cols].values.astype(np.float32)
lds_train_weights= compute_lds_weights(training_labels_for_lds)

print(f"now the LDS weights have been computed for the {len(lds_train_weights)} training samples")
print("their summary stats:")
print(f"min weight= {lds_train_weights.min():.4f} & max weight= {lds_train_weights.max():.4f}. Also mean weight= {lds_train_weights.mean():.4f} (which I'm expecting is ~1.0)")

now the LDS weights have been computed for the 1298 training samples
their summary stats:
min weight= 0.2697 & max weight= 14.9363. Also mean weight= 1.0000 (which I'm expecting is ~1.0)


In [12]:
# now the DeBERTa-v3 tokeniser is setup
tokenizer= AutoTokenizer.from_pretrained(threat_only_aux_hyperparams["MODEL_NAME"], use_fast=False)

### Dataset wrappers and data loaders

Below I define two pytorch dataset wrappers. The first is `SecuritisationDataset` (identical to my baseline), and that holds the tokenised gold sentences alongside the three normalised dimension labels and (where its applicable) the LDS weights I precomputed. The second is `WeakOnlyDataset` (new in this variant) and its job is to wrap the deduplicated weak pool by holding the lexicon-derived pseudo-labels but not gold labels. The training loaders carry the LDS weights for gold but not weak. Validation & test loaders are uniformly weighted, which I did intentionally to keep their evaluation unbiased.

In [13]:
# this wraps a df of sentences + labels (+ optional LDS weights) so the DataLoader can batch them all up for the model
class SecuritisationDataset(Dataset):
    def __init__(self, source_df, tokenizer, max_len, sample_weights=None):
        self.sentences= source_df["sentence_text"].tolist()
        self.labels= source_df[dim_score_cols].values
        self.tokenizer= tokenizer
        self.max_len= max_len
        self.sample_weights= sample_weights # as mentioned above, I put none for val/test but do populate this for train
    def __len__(self):
        return len(self.sentences)
    def __getitem__(self, sample_idx):
        encoded= self.tokenizer(self.sentences[sample_idx],padding="max_length", truncation=True, max_length=self.max_len,return_tensors="pt")
        batch_item= {key: tensor.squeeze(0) for key, tensor in encoded.items()}
        batch_item["labels"]= torch.tensor(self.labels[sample_idx], dtype=torch.float)
        # I only attatched the LDS weight when sample_weights is provided, i.e. for the training set
        if self.sample_weights is not None:
            batch_item["weight"]= torch.tensor(self.sample_weights[sample_idx], dtype=torch.float)
        return batch_item


class WeakOnlyDataset(Dataset):
    # wraps the weak pool sentences w/ just the threat lexicon's pseudo-label. No gold labels here & no urgency/justification weak scores either (this variant only consumes threat)
    def __init__(self, source_df, tokenizer, max_len, threat_weak_col):
        self.sentences= source_df["sentence_text"].tolist()
        self.weak_threat_scores= source_df[threat_weak_col].values
        self.tokenizer= tokenizer
        self.max_len= max_len
    def __len__(self):
        return len(self.sentences)
    def __getitem__(self, sample_idx):
        encoded= self.tokenizer(self.sentences[sample_idx], padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        batch_item= {key: tensor.squeeze(0) for key, tensor in encoded.items()}
        batch_item["weak_threat"]= torch.tensor(self.weak_threat_scores[sample_idx], dtype=torch.float) # scalar for the threat pseudo-label
        return batch_item

In [14]:
# now I build the 4 datasets I need (3 gold, 1 weak)
max_len= threat_only_aux_hyperparams["MAX_LEN"]

# optional weak-pool subsampling. With WEAK_POOL_SUBSAMPLE_FRACTION=1.0 (default) I use the full pool, since the 6:1 batch ratio rate-limits exposure naturally
if threat_only_aux_hyperparams["WEAK_POOL_SUBSAMPLE_FRACTION"] < 1.0:
    n_weak_subsample= int(len(weak_only_df) * threat_only_aux_hyperparams["WEAK_POOL_SUBSAMPLE_FRACTION"])
    weak_pool_for_training= weak_only_df.sample(n=n_weak_subsample, random_state=random_seed).reset_index(drop=True)
    print(f"weak pool subsampled to {n_weak_subsample}/{len(weak_only_df)} sentences")
else:
    weak_pool_for_training= weak_only_df
    print(f"using the full weak pool of {len(weak_only_df)} sentences")

train_dataset= SecuritisationDataset(train_df, tokenizer, max_len, sample_weights=lds_train_weights)
val_dataset= SecuritisationDataset(val_df, tokenizer, max_len)
test_dataset= SecuritisationDataset(test_df, tokenizer, max_len)
weak_train_dataset= WeakOnlyDataset(weak_pool_for_training, tokenizer, max_len, "threat_weak_score")

# I had to do this because physical batch was small to fit the GPU. 
# Instead I use gradient accumulation in the training loop to restore the effective batch size.
physical_batch_size= threat_only_aux_hyperparams["PHYSICAL_BATCH_SIZE"]
train_loader= DataLoader(train_dataset, batch_size=physical_batch_size, shuffle=True)
val_loader= DataLoader(val_dataset, batch_size=16)
test_loader= DataLoader(test_dataset, batch_size=16)
weak_train_loader= DataLoader(weak_train_dataset, batch_size=physical_batch_size, shuffle=True)

print(f"loaders ready - gold train: {len(train_loader)} batches | weak train: {len(weak_train_loader)} batches | val: {len(val_loader)} batches | test: {len(test_loader)} batches")

using the full weak pool of 43644 sentences
loaders ready - gold train: 163 batches | weak train: 5456 batches | val: 15 batches | test: 15 batches


### Mean pooling over non-padding tokens

Now I pool the encoder outputs by mean-aggregating the hidden states over non-padding token positions instead of just using the CLS token alone. [3] showed this produces richer sentence representations for tasks where meaning is distributed across the sentence, which very much applies here as securitising framing cues are typically spread across a clause instead of concentrated on a single token.

In [15]:
# this mean pools over non-padding token embeddings
# token_embeddings is (B, T, H), attention_mask is (B, T) with 1 for real tokens & 0 for padding
# note: I read the full code implementation for pooling on the huggingface repo, learning from it and taking the bit I needed to apply here. You can find it at: https://github.com/huggingface/sentence-transformers/blob/main/sentence_transformers/sentence_transformer/modules/pooling.py
def mean_pool(token_embeddings, attention_mask):
    mask_expanded= attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed_embeddings= torch.sum(token_embeddings * mask_expanded, dim=1) # I 0 out padding positions before summing
    real_token_counts= torch.clamp(mask_expanded.sum(dim=1), min=1e-9) # & divide by the number of real tokens
    return summed_embeddings / real_token_counts

### Multi-sample dropout regression head

In my model, each dimension is predicted by an independent two-layer MLP head that uses multi-sample dropout [4]. During training I apply K(=2) independent dropout masks to the same pooled representation to produce K predictions that are averaged before computing loss. The idea of this was to act as an efficient implicit ensemble regulariser, which matters given how small the gold training set is. I use a sigmoid activation at the output to bound predictions to [0, 1]. This head class is shared across both the gold and the auxiliary head positions in this variant - the only difference between gold and aux heads is what they're trained on (gold labels vs. weak pseudo-labels), not how they're built.

In [16]:
# This is my per-dimension regression head that I use in my model.
# In this variant it has multiple instances - 3 for gold heads (one per securitisation dimension) plus the additional auxiliary head(s) used for the weak pool training pathway
# my architecture is LayerNorm -> [K=2 parallel dropout passes -> Linear -> GELU -> Linear] -> average across the K passes -> Sigmoid

# the multi-sample dropout idea comes from Inoue [4], with its idea being that instead of doing 1 forward pass per training step with 1 random dropout mask, you do K passes with K different masks & average the predictions before computing loss
# I had to chose K=2 because of the limited 10GB NCC GPU I get, but I learnt that additional K is mainly diminishing returns anyway
# the 2-layer MLP shape (hidden -> hidden//2 -> 1) uses [5]'s standard MT-DNN head pattern
# the logic behind this choice is that the bottleneck + non-linearity gives the head capacity to learn a dimension-specific transformation of the shared encoder representation, 
# whereas if I just had a single linear layer it would only give it linear capacity & all 3 dim heads would end up sharing pretty much the same projection geometry
# GELU specifically because that's what DeBERTa uses internally, so I'm staying consistent with the encoder's own activation
# The LayerNorm early in the function is from [5]. I use it because it stabilises the distribution of encoder outputs going into the heads, which matters here because all heads pull on the same shared encoder during training 
# & without that normalisation 1 dim's gradient could shift the encoder activation scale enough to mess with the others

# note: I learnt some of the theory behind the code for this from the link, and adapted the idea to my own context: https://www.kaggle.com/code/doanquanvietnamca/multi-dropout-kfold-with-roberta-on-gpu 
class MultiSampleDropoutHead(nn.Module):
    def __init__(self, encoder_hidden_size, dropout_probability, num_dropout_samples):
        super().__init__()
        self.num_dropout_samples= num_dropout_samples
        self.ln= nn.LayerNorm(encoder_hidden_size)

        # the K independent dropouts each sample a new random mask on each forward call
        # I could've just used 1 dropout module & called it K times in the loop (works the same since each call samples a new mask), 
        # but having K separate modules makes my architecture a bit more explicit when I print the model & keeps K easy to inspect from outside
        self.dropout_modules= nn.ModuleList([nn.Dropout(dropout_probability) for _ in range(num_dropout_samples)])

        # the 2-layer MLP is down-projected to hidden/2, GELU then projected to a scalar logit
        self.fc1= nn.Linear(encoder_hidden_size, encoder_hidden_size // 2)
        self.activation= nn.GELU()
        self.fc2= nn.Linear(encoder_hidden_size // 2, 1)

    def forward(self, encoder_pooled_output):
        # LayerNorm only happens once since it's not random, there's no point redoing it inside the K-pass loop below
        layer_normed_input= self.ln(encoder_pooled_output)

        if self.training: # training, not inference, so model goes ahead and runs the training logic
            # multi-sample dropout w/ those K forward passes through the MLP with K different dropout masks. They collect each pass's logit, stack & average
            logits_from_each_dropout_sample= []
            for kth_dropout_module in self.dropout_modules:
                input_after_kth_mask= kth_dropout_module(layer_normed_input)
                logits_from_each_dropout_sample.append(self.fc2(self.activation(self.fc1(input_after_kth_mask))))
            averaged_logits= torch.stack(logits_from_each_dropout_sample, dim=0).mean(dim=0) # averaging happens pre-sigmoid not post - the difference is tiny in practice for K=2 but pre-sigmoid gives cleaner gradients
        else: # inference on corpus, model's been trained
            # at inference time self.eval() at the model level disables dropout entirely, so doing K passes would just produce K identical results. Thus 1 deterministic forward pass is what I do here
            averaged_logits= self.fc2(self.activation(self.fc1(layer_normed_input)))

        # finally, the sigmoid clamps the output to [0, 1] to match my normalised label range
        return torch.sigmoid(averaged_logits)

### Multi-head securitisation model with single auxiliary head

Now I put the encoder, mean pooling, the three independent gold regression heads and the single new auxiliary threat head into one `nn.Module`. This follows the canonical multi-task learning framework of Caruana [10] which Liu et al. [5] extended to transformer architectures. The gold heads are the ones I keep at inference and that produce my final predictions; the auxiliary head exists purely for the training-time gradient signal from the weak pool's threat column, and is discarded at inference. The forward pass returns the gold heads' logits unconditionally, and (only when training or when explicitly requested via `return_aux=True`) also returns the aux head's logit. The gold-head loss path still uses the dimension-weighted SmoothL1 from baseline; the auxiliary loss is computed externally in the training loop because the weak pool has no gold labels.

In [17]:
# this is the shared DeBERTa-v3-base encoder with 3 gold heads + 1 auxiliary threat head
# the gold heads are kept at inference; the auxiliary head is discarded so the model's output signature stays identical to baseline
class SemiSupervisedThreatOnlyAuxModel(nn.Module):
    def __init__(self, model_name, head_dropout, num_dropout_samples):
        super().__init__()
        # shared encoder, which is the whole backbone of my model & gets fine-tuned on both gold and auxiliary signals during training
        self.encoder= AutoModel.from_pretrained(model_name, use_safetensors=True)
        hidden_size= self.encoder.config.hidden_size
        self.pooled_dropout= nn.Dropout(head_dropout) # this is the dropout I apply to the pooled representation before it hits any head

        # 3 gold heads, identical to my baseline model
        self.threat_head= MultiSampleDropoutHead(hidden_size, head_dropout, num_dropout_samples)
        self.urgency_head= MultiSampleDropoutHead(hidden_size, head_dropout, num_dropout_samples)
        self.justification_head= MultiSampleDropoutHead(hidden_size, head_dropout, num_dropout_samples)

        # single auxiliary head, trained on the threat lexicon's pseudo-labels & discarded at inference
        # urgency & justification have no auxiliary supervision in this variant - this is the deliberate restriction the threat-only design tests
        self.aux_threat_head= MultiSampleDropoutHead(hidden_size, head_dropout, num_dropout_samples)

    def forward(self, input_ids, attention_mask, labels=None, return_aux=False):
        encoder_outputs= self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled= mean_pool(encoder_outputs.last_hidden_state, attention_mask) # mean pooling over non-padding tokens (not standard CLS-only pooling)
        pooled= self.pooled_dropout(pooled)

        # the 3 gold heads, each returning a [0, 1] score from its internal sigmoid
        threat_logit= self.threat_head(pooled)
        urgency_logit= self.urgency_head(pooled)
        justification_logit= self.justification_head(pooled)
        stacked_logits= torch.cat([threat_logit, urgency_logit, justification_logit], dim=1) # (B, 3) gold predictions

        result= {"logits": stacked_logits, "loss": None}

        # I only run the aux head during training (or when explicitly asked via return_aux=True), so at eval-time inference is identical to baseline & doesn't waste compute on a head I'm about to discard
        if self.training or return_aux:
            result["aux_threat_logits"]= self.aux_threat_head(pooled) # (B, 1) threat-only auxiliary prediction

        # gold-loss path is identical to baseline; this is what evaluate-helper uses
        # the training loop builds its own loss externally so it can include LDS weighting, R-Drop & the aux loss
        if labels is not None:
            dim_loss_weights= torch.tensor([threat_only_aux_hyperparams["LOSS_WEIGHT_THREAT"], threat_only_aux_hyperparams["LOSS_WEIGHT_URGENCY"], threat_only_aux_hyperparams["LOSS_WEIGHT_JUSTIFICATION"]], device=stacked_logits.device).unsqueeze(0) # (1, 3) broadcasts across batch
            per_dim_loss= nn.functional.smooth_l1_loss(stacked_logits, labels.float(), reduction="none") # (B, 3)
            result["loss"]= (per_dim_loss * dim_loss_weights).mean()

        return result

### Initialisation, LLRD optimiser, scheduler and AMP

In this next part I instantiate the model I defined above and configure its layer-wise learning rate decay (LLRD) so that each transformer layer gets a progressively smaller learning rate the deeper it is in the stack. The idea is that it preserves the general linguistic knowledge that the lower layers have whilst letting the upper layers & task-specific heads adapt more freely. I pair the optimiser with a linear warmup followed by cosine decay, which is inspired by [8]'s configuration recomendations, and that stabilises early training + anneals smoothly toward zero. I also have mixed-precision training, which uses a dynamic-scale `GradScaler` to stop any gradient underflow's from float16. The auxiliary head is added to the head parameter groups at the same base learning rate as the gold heads since they're both trained from scratch.

In [18]:
# instantiating the model & giving it the GPU
head_dropout= threat_only_aux_hyperparams["HEAD_DROPOUT"]
num_dropout_samples= threat_only_aux_hyperparams["NUM_DROPOUT_SAMPLES"]
model= SemiSupervisedThreatOnlyAuxModel(threat_only_aux_hyperparams["MODEL_NAME"], head_dropout, num_dropout_samples)
model.to(device)
model= model.float()
# I used [7] & [8] for fine-tuning guidance here
base_lr= threat_only_aux_hyperparams["BASE_LR"]
llrd_factor= threat_only_aux_hyperparams["LLRD_FACTOR"]

# the 3 gold heads + the 1 auxiliary threat head all get the full base LR because they're all trained from scratch
three_sec_head_param_groups= [{"params": model.threat_head.parameters(), "lr": base_lr},
                              {"params": model.urgency_head.parameters(), "lr": base_lr},
                              {"params": model.justification_head.parameters(), "lr": base_lr},
                              {"params": model.aux_threat_head.parameters(), "lr": base_lr}]

# these transformer layers get descending LRs from top layer to bottom
# pos_of_layer=0 is the encoder layer at the very top (which gets the highest LR after the heads)
layers_of_sec_encoder= []
if hasattr(model.encoder, "encoder") and hasattr(model.encoder.encoder, "layer"): # i.e. if it's an encoder layer, pull it out into a lst
    layers_of_sec_encoder= list(model.encoder.encoder.layer)
layer_param_groups= []
cnt_of_sec_encoder_layers= len(layers_of_sec_encoder)
for pos_of_layer, encoder_layer in enumerate(reversed(layers_of_sec_encoder)):
    decayed_lr= base_lr * (llrd_factor ** (pos_of_layer + 1)) # this is where I apply the decay
    layer_param_groups.append({"params": encoder_layer.parameters(), "lr": decayed_lr})

# I give embeddings & any other remaining encoder params (e.g. position embeddings) the smallest LR
remaining_encoder_params= [sec_model_param for name, sec_model_param in model.encoder.named_parameters() 
                           if not any(f"layer.{layer_num}." in name for layer_num in range(cnt_of_sec_encoder_layers))]
embedding_lr= base_lr * (llrd_factor ** (cnt_of_sec_encoder_layers + 1))
layer_param_groups.append({"params": remaining_encoder_params, "lr": embedding_lr})
all_param_groups= three_sec_head_param_groups + layer_param_groups
sec_optimiser= torch.optim.AdamW(all_param_groups, weight_decay=0.01)
print(f"successfull built the LLRD optimiser with {len(all_param_groups)} param groups")
print(f"the top layer LR is {base_lr * llrd_factor:.2e} & embedding LR: {embedding_lr:.2e}")

# now for linear warmup [9] followed by cosine decay [10]
# I learnt about the code behind the linear warmup / cosing decay pairing here, and adapted it to my model: https://github.com/huggingface/transformers/blob/main/src/transformers/optimization.py
training_epochs= threat_only_aux_hyperparams["TRAINING_EPOCHS"]
warmup_proportion= threat_only_aux_hyperparams["WARMUP_PROPORTION"]
overall_steps_ran= len(train_loader) * training_epochs
warmup_steps= int(overall_steps_ran * warmup_proportion)
def lr_decay(current_step): # so its a linear ramp from 0 to 1 during warmup then cosine decay from 1 to 0 over the rest of training
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    cosine_progress= float(current_step - warmup_steps) / float(max(1, overall_steps_ran - warmup_steps))
    return max(0.0, 0.5 * (1.0 + np.cos(np.pi * cosine_progress)))

scheduler= torch.optim.lr_scheduler.LambdaLR(sec_optimiser, lr_decay)
print(f"scheduler is setup, and it has linear warmup for {warmup_steps} steps then cosine decay over {overall_steps_ran} steps total")

# I use the AMP GradScaler, which keeps a dynamic loss scale to prevent any float16 gradient underflows [11]
scaler= GradScaler()

successfull built the LLRD optimiser with 17 param groups
the top layer LR is 1.80e-05 & embedding LR: 5.08e-06
scheduler is setup, and it has linear warmup for 244 steps then cosine decay over 2445 steps total


In [19]:
# this is my validation-time helper, which returns mean SmoothL1 loss & per-dimension Pearson r over the full dataloader
# I use it inside the training loop for early-stopping decisions & progress reporting
# Pearson r is chosen specifically because it captures rank-order alignment with the gold labels
# this only ever uses the gold heads (the auxiliary heads aren't called - the model's forward pass skips them when self.training is False & return_aux defaults to False), so evaluation is identical to baseline
def run_evaluation_using_pearson(sec_regr_model, loader_of_batches, device):
    sec_regr_model.eval()
    total_model_loss= 0
    all_preds, all_model_labels= [], []
    with torch.no_grad():
        for batch_of_labels in loader_of_batches:
            input_ids= batch_of_labels["input_ids"].to(device)
            attention_mask= batch_of_labels["attention_mask"].to(device)
            labels_in_batch= batch_of_labels["labels"].to(device)
            sec_regr_outputs= sec_regr_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels_in_batch)
            total_model_loss += sec_regr_outputs["loss"].item()
            all_preds.append(sec_regr_outputs["logits"].cpu().numpy())
            all_model_labels.append(labels_in_batch.cpu().numpy())

    avg_loss= total_model_loss / len(loader_of_batches)
    all_preds, all_model_labels= np.concatenate(all_preds, axis=0), np.concatenate(all_model_labels, axis=0) # both (N, 3)

    per_dim_pearson= {} # now I get the pearson score for all 3 dims
    for dim_pos, dim_name in enumerate(dim_score_cols):
        r_val, _= pearsonr(all_preds[:, dim_pos], all_model_labels[:, dim_pos])
        per_dim_pearson[dim_name]= r_val
    return avg_loss, per_dim_pearson

### Training loop

This is where the semi-supervised mechanism actually takes effect. The below code is the same as my baseline model (i.e. it has two-pass R-Drop, LDS-weighted SmoothL1, AMP, gradient accumulation, early stopping and all other things), but it adds a weak-pool pathway I setup to run alongside each gold batch. For every gold batch I run the baseline's loss computation, then run `WEAK_BATCHES_PER_GOLD_BATCH` (=6) weak batches through the single auxilary threat head to compute a SmoothL1 loss between the aux predictions and the lexicon-derived pseudo-labels. I scale the weak-loss term by `LAMBDA_AUX_WEAK` (=0.5) and accumulate it in the same gradient buffer as the gold loss before each optimiser step. Because the weak loader has more batches than the gold loader I cycle it using `itertools.cycle` so that it wraps around when it runs out. R-Drop is kept gold-only as the weak path has no gold labels, and so is LDS. The validation I run is on gold-only too, and its identical to baseline.

In [20]:
# this is my main training loop. It uses R-Drop [12] + LDS-weighted SmoothL1 + gradient accumulation + early stopping, w/ mixed precision training ideas [11] incorporated, plus the additional weak pathway feeding the auxiliary head(s)
# note: for R-Drop, I used the official paper repo for inspiration, but adapted it from their bidirectional KL to MSE, as they worked w/ classifiers but my model is a regression one.
# You can find that at: https://github.com/dropreg/R-Drop
# Also, I learnt about AMP from: https://docs.pytorch.org/tutorials/recipes/recipes/amp_recipe.html
# & the theory behind early stopping on: https://medium.com/@vrunda.bhattbhatt/a-step-by-step-guide-to-early-stopping-in-tensorflow-and-pytorch-59c1e3d0e376
# the auxiliary supervision design follows [10] (Caruana 1997) & [11] (Ratner et al. Data Programming): https://github.com/HazyResearch/snorkel
best_val_loss= float("inf") # this is something I keep track of for the early stopping I setup
epochs_with_no_improvement= 0
best_model_state= None
training_epochs= threat_only_aux_hyperparams["TRAINING_EPOCHS"]
grad_accum_steps= threat_only_aux_hyperparams["GRAD_ACCUM_STEPS"]
grad_clip_max_norm= threat_only_aux_hyperparams["GRAD_CLIP_MAX_NORM"]
lambda_rdrop= threat_only_aux_hyperparams["LAMBDA_RDROP"]
lambda_aux_weak= threat_only_aux_hyperparams["LAMBDA_AUX_WEAK"]
weak_batches_per_gold= threat_only_aux_hyperparams["WEAK_BATCHES_PER_GOLD_BATCH"]
early_stopping_patience= threat_only_aux_hyperparams["EARLY_STOPPING_PATIENCE"]

# 1st I precompute the dim-weight tensors so I don't have to rebuild them every batch
dim_loss_weights= torch.tensor([threat_only_aux_hyperparams["LOSS_WEIGHT_THREAT"], threat_only_aux_hyperparams["LOSS_WEIGHT_URGENCY"], threat_only_aux_hyperparams["LOSS_WEIGHT_JUSTIFICATION"]],device=device).unsqueeze(0) # (1, 3) broadcasts across batch
# threat-only variant doesn't need a weak_dim_weights tensor since there's only 1 weak dimension

# the weak loader has more batches than the gold loader, so I cycle it w/ itertools.cycle so it wraps round when exhausted
weak_iter= iter(itertools.cycle(weak_train_loader))

# now repeated epochs are carried out to train the model
for epoch_idx in range(training_epochs): 
    model.train()
    total_train_loss= 0
    total_weak_loss= 0
    sec_optimiser.zero_grad() # its zeroed once per accumulation cycle, not once per batch

    for curr_batch_num, batch in enumerate(tqdm(train_loader)):
        # Important note: Gold pathway is this bit
        # this is identical to my baseline gold-loss computation
        input_ids, attention_mask, labels= batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["labels"].to(device)
        with autocast(device_type="cuda"): # I training it on cpu but it takes ages, so if you're running it, make sure you have GPU availble
            # 1st stochastic forward pass
            outputs_pass_1= model(input_ids=input_ids, attention_mask=attention_mask)
            logits_pass_1= outputs_pass_1["logits"] # (B, 3)

            # 2nd stochastic forward pass w/ different dropout mask
            outputs_pass_2= model(input_ids=input_ids, attention_mask=attention_mask) # model.train() active so each call automatically samples a fresh mask 
            logits_pass_2= outputs_pass_2["logits"] # (B, 3)

            # primary loss is computed on the averaged predictions, which is the same info as 1 pass but w/ reduced variance
            averaged_logits= (logits_pass_1 + logits_pass_2) / 2.0
            per_sample_loss= nn.functional.smooth_l1_loss(averaged_logits, labels.float(), reduction="none") # (B, 3)
            per_sample_loss= per_sample_loss * dim_loss_weights # applying dimension-specific weights
            if "weight" in batch:
                lds_weight= batch["weight"].to(device).unsqueeze(1) # (B, 1)
                primary_loss= (per_sample_loss * lds_weight).mean()
            else:
                primary_loss= per_sample_loss.mean()

            # R-Drop consistency term, which is the MSE between the 2 stochastic predictions
            # I use MSE instead of KL-divergence because this is regression not classification, but the latter is what the original paper uses because theirs *is* classification
            rdrop_loss= nn.functional.mse_loss(logits_pass_1, logits_pass_2)
            # I divide the combined loss by accumulation steps so the summed gradients match a full-batch step
            combined_loss= (primary_loss + lambda_rdrop*rdrop_loss)/grad_accum_steps
        scaler.scale(combined_loss).backward()
        total_train_loss += combined_loss.item()*grad_accum_steps

        # Important note: Weak pathway is this bit
        # for each gold batch I run WEAK_BATCHES_PER_GOLD_BATCH weak batches through the threat-only aux head
        # the weak loader is cycled so it wraps round when exhausted
        # I don't have R-Drop here because R-Drop is a regularisation term over gold labels which I don't have on the weak pool
        for _ in range(weak_batches_per_gold):
            weak_batch= next(weak_iter)
            w_input_ids= weak_batch["input_ids"].to(device)
            w_attention_mask= weak_batch["attention_mask"].to(device)
            w_weak_threat_targets= weak_batch["weak_threat"].to(device).unsqueeze(1) # threat pseudo-labels only, w/ shape (B, 1) 

            with autocast(device_type="cuda"):
                w_outputs= model(input_ids=w_input_ids, attention_mask=w_attention_mask)
                w_aux_threat_logits= w_outputs["aux_threat_logits"] # (B, 1) auxiliary threat prediction

                # SmoothL1 between aux threat prediction & threat pseudo-label, no dim weighting since this is single-dimensional
                primary_weak_loss= nn.functional.smooth_l1_loss(w_aux_threat_logits, w_weak_threat_targets, reduction="mean")

                weak_loss= (lambda_aux_weak * primary_weak_loss) / grad_accum_steps
            scaler.scale(weak_loss).backward()
            total_weak_loss += weak_loss.item() * grad_accum_steps

        # this is my optimiser step
        # both the gold pathway's gradient & the WEAK_BATCHES_PER_GOLD_BATCH worth of weak gradients are now accumulated in the same buffer ready for the optimiser step
        # stepping happens once every 'grad_accum_steps' batches (or on the final batch of the epoch)
        is_accumulation_boundary= ((curr_batch_num+ 1) % grad_accum_steps == 0) or ((curr_batch_num + 1) == len(train_loader))
        if is_accumulation_boundary:
            # this is where I do gradient clipping to prevent large updates from corrupting pretrained weights [13]
            scaler.unscale_(sec_optimiser)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_max_norm)
            scaler.step(sec_optimiser)
            scaler.update()
            scheduler.step() # LR gets stepped each time the optimiser does
            sec_optimiser.zero_grad()

    avg_train_loss= total_train_loss / len(train_loader)
    avg_weak_loss= total_weak_loss / (len(train_loader) * weak_batches_per_gold)
    avg_val_loss, val_pearson= run_evaluation_using_pearson(model, val_loader, device)

    print(f"\nepoch {epoch_idx+1}")
    print(f"gold train loss: {avg_train_loss:.4f} | weak aux loss: {avg_weak_loss:.4f}")
    print(f"val loss: {avg_val_loss:.4f}")
    for name_of_sec_dimension, pearson_r_value in val_pearson.items():
        print(f"- val Pearson r (for {name_of_sec_dimension}): {pearson_r_value:.4f}")
    print(f"- current LR (head) is {scheduler.get_last_lr()[0]:.2e}")

    if avg_val_loss < best_val_loss:
        best_val_loss= avg_val_loss
        epochs_with_no_improvement= 0
        best_model_state= copy.deepcopy(model.state_dict())
        print(f"new best val loss reached!! that checkpoint is now saved in memory")
    else:
        epochs_with_no_improvement += 1
        print(f"there's been no improvement for {epochs_with_no_improvement}/{early_stopping_patience} patience epochs")
        if epochs_with_no_improvement >= early_stopping_patience:
            print(f"\nearly stopping has now been triggered, specifically at epoch {epoch_idx+1}, & best weights are now restored.")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("\nbest model weights restored")

100%|██████████| 163/163 [04:35<00:00,  1.69s/it]



epoch 1
gold train loss: 0.1078 | weak aux loss: 0.0337
val loss: 0.0819
- val Pearson r (for threat_score): 0.0395
- val Pearson r (for urgency_score): 0.0064
- val Pearson r (for justification_score): 0.0636
- current LR (head) is 6.72e-06
new best val loss reached!! that checkpoint is now saved in memory


100%|██████████| 163/163 [04:34<00:00,  1.68s/it]



epoch 2
gold train loss: 0.0857 | weak aux loss: 0.0055
val loss: 0.0514
- val Pearson r (for threat_score): 0.4044
- val Pearson r (for urgency_score): 0.2991
- val Pearson r (for justification_score): 0.3795
- current LR (head) is 1.34e-05
new best val loss reached!! that checkpoint is now saved in memory


100%|██████████| 163/163 [04:34<00:00,  1.69s/it]


...


epoch 8
gold train loss: 0.0163 | weak aux loss: 0.0018
val loss: 0.0159
- val Pearson r (for threat_score): 0.7699
- val Pearson r (for urgency_score): 0.5276
- val Pearson r (for justification_score): 0.5524
- current LR (head) is 1.83e-05
there's been no improvement for 3/5 patience epochs


100%|██████████| 163/163 [04:30<00:00,  1.66s/it]



epoch 9
gold train loss: 0.0144 | weak aux loss: 0.0017
val loss: 0.0151
- val Pearson r (for threat_score): 0.7447
- val Pearson r (for urgency_score): 0.5524
- val Pearson r (for justification_score): 0.5154
- current LR (head) is 1.76e-05
there's been no improvement for 4/5 patience epochs


100%|██████████| 163/163 [04:30<00:00,  1.66s/it]



epoch 10
gold train loss: 0.0118 | weak aux loss: 0.0017
val loss: 0.0142
- val Pearson r (for threat_score): 0.7830
- val Pearson r (for urgency_score): 0.5660
- val Pearson r (for justification_score): 0.5292
- current LR (head) is 1.68e-05
there's been no improvement for 5/5 patience epochs

early stopping has now been triggered, specifically at epoch 10, & best weights are now restored.

best model weights restored


### Held-out test set

This is my evaluation of the (best-checkpoint) model on the held-out test set with the exact same helper used at validation time, which gives me a quick per-dimension Pearson reading alongside the overall loss. I do a full metric evaluation below though, so see that for further details on my model's performance. Importantly, evaluation only uses the gold heads (the auxiliary head(s) aren't called at inference, so the model's forward pass just skips them when not training). Through this, evaluation is structurally identical to my baseline.

In [21]:
avg_test_loss, test_pearson= run_evaluation_using_pearson(model, test_loader, device)
print(f"test loss was {avg_test_loss:.4f}")
for dim_name, r_value in test_pearson.items():
    print(f"- test Pearson r ({dim_name}): {r_value:.4f}")

test loss was 0.0180
- test Pearson r (threat_score): 0.7965
- test Pearson r (urgency_score): 0.4948
- test Pearson r (justification_score): 0.5021


### Comprehensive test-set evaluation

Now I calculate all the per-dimension regression metrics I need on the test set. Specifically these are Pearson r, Spearman rho, MSE, RMSE, and MAE, followed by their macro-averages, as those are all the metric's I've seen be used in multiple paper contexts for reporting similar results. I chose Spearman rho as my primary metric though because it captures rank-order alignment with the gold labels, which matters more than an absolute calibration for the securitisation-specific nature of my model. I also compute Quadratic Weighted Cohen's Kappa between predictions & gold Likert classes as a model-human agreement measurement for my report. The test predictions and labels are saved to disk so the cross-model comparison scripts can load them for paired statistical significance tests against the baseline.

In [22]:
# here I'm running a fresh inference pass over the test set & collecting predictions + labels for downstream metric computation
model.eval()
test_preds_batches= []
test_labels_batches= []

with torch.no_grad():
    for batch in test_loader:
        input_ids= batch["input_ids"].to(device)
        attention_mask= batch["attention_mask"].to(device)
        labels_cpu= batch["labels"] # I keep it on cpu for the numpy conversion

        outputs= model(input_ids=input_ids, attention_mask=attention_mask)
        test_preds_batches.append(outputs["logits"].cpu().numpy())
        test_labels_batches.append(labels_cpu.numpy())

test_preds= np.vstack(test_preds_batches) # (N, 3), [0, 1]
test_labels= np.vstack(test_labels_batches) # (N, 3), [0, 1]
print(f"collected {len(test_preds)} test predictions across {len(dim_score_cols)} dimensions")

collected 225 test predictions across 3 dimensions


In [23]:
warnings.filterwarnings("ignore", category=RuntimeWarning)
per_dim_results= {}
print("per-dimension evaluation metrics")
print("")

for cnt_of_dim, dim_name in enumerate(dim_score_cols):
    dim_preds, dim_labels= test_preds[:, cnt_of_dim], test_labels[:, cnt_of_dim]
    valid_mask= ~(np.isnan(dim_preds) | np.isnan(dim_labels)) # I 1st filter out any nan rows that could slip through
    dim_preds, dim_labels= dim_preds[valid_mask], dim_labels[valid_mask]

    # now for all the metrics, which I just call their corresponding library function for
    pearson_r, _= pearsonr(dim_labels, dim_preds)
    spearman_rho, _= spearmanr(dim_labels, dim_preds)
    dim_mse= mean_squared_error(dim_labels, dim_preds)
    dim_rmse= float(np.sqrt(dim_mse))
    dim_mae= mean_absolute_error(dim_labels, dim_preds)
    per_dim_results[dim_name]= {"pearson": pearson_r, "spearman": spearman_rho,
        "mse": dim_mse, "rmse": dim_rmse, "mae": dim_mae, "preds": dim_preds, "labels": dim_labels} # I store those for downstream QWK
    print(f"\n{dim_name}")
    print(f"- Pearson r: {pearson_r:.4f}")
    print(f"- Spearman rho: {spearman_rho:.4f}")
    print(f"- MSE: {dim_mse:.4f}")
    print(f"- RMSE: {dim_rmse:.4f}")
    print(f"- MAE: {dim_mae:.4f}")

# now I find the macro-averages across dimensions, which I use in my paper alongside key metrics above to summarise overall performance
print("")
print("macro-avg across dimensions:")
for metric_name in ["pearson", "spearman", "mse", "rmse", "mae"]:
    model_metric_vals= [per_dim_results[dim][metric_name] for dim in dim_score_cols if dim in per_dim_results]
    if model_metric_vals:
        print(f"- macro-avg {metric_name}: {float(np.mean(model_metric_vals)):.4f}")

per-dimension evaluation metrics


threat_score
- Pearson r: 0.7965
- Spearman rho: 0.6060
- MSE: 0.0195
- RMSE: 0.1396
- MAE: 0.0958

urgency_score
- Pearson r: 0.4948
- Spearman rho: 0.3732
- MSE: 0.0232
- RMSE: 0.1522
- MAE: 0.0946

justification_score
- Pearson r: 0.5021
- Spearman rho: 0.3087
- MSE: 0.0281
- RMSE: 0.1676
- MAE: 0.1145

macro-avg across dimensions:
- macro-avg pearson: 0.5978
- macro-avg spearman: 0.4293
- macro-avg mse: 0.0236
- macro-avg rmse: 0.1531
- macro-avg mae: 0.1016


In [24]:
# & the quadratic weighted Cohen's Kappa [15] that I use as a model-human agreement measurement
# I discretise continuous predictions to the nearest Likert class {0, 1/3, 2/3, 1} before I calculate the kappa
# note: this is NOT inter-annotator agreement, as I am the only human annotator that produced the gold labels, but as I
# don't have anybody to compare agreement with, this is the best measurment I can give for my dissertation.
# I've been clear on this for my project report, and highlighted having another annotator as being something needed in future work.
ordinal_model_label_classes= np.array([0.0, 1/3, 2/3, 1.0])
def put_in_nearest_likert_class(continuous_array, classes=ordinal_model_label_classes):
    # maps continuous [0, 1] values to the nearest discrete Likert class
    return classes[np.argmin(np.abs(continuous_array[:, None] - classes[None, :]), axis=1)]

print("Quadratic Weighted Cohen's Kappa (for model vs human annotator agreement checking)")
print("")

per_dim_kappas= []
for cnt_of_dim, dim_name in enumerate(dim_score_cols):
    if dim_name not in per_dim_results:
        continue
    preds_discrete, labels_discrete= put_in_nearest_likert_class(per_dim_results[dim_name]["preds"]), put_in_nearest_likert_class(per_dim_results[dim_name]["labels"])

    # here I scale back up by 3 to get integer Likert classes {0, 1, 2, 3} so it works w/ sklearn
    dim_kappa= cohen_kappa_score(np.round(labels_discrete*3).astype(int), np.round(preds_discrete*3).astype(int), weights="quadratic",labels=[0, 1, 2, 3])
    per_dim_kappas.append(dim_kappa)
    print(f"{dim_name} QWK= {dim_kappa:.4f}")

if per_dim_kappas:
    print(f"\nAnd macro-avg QWK= {float(np.mean(per_dim_kappas)):.4f}")

# I print the Landis & Koch [16] benchmarks for interpreting kappa values, which is what I'm going by to gauge how aligned my model's predictions are with my own
print("\nThe Landis & Koch benchmarks are:")
print("<0.20=slight, 0.21-0.40=fair, 0.41-0.60=moderate, 0.61-0.80=substantial, >0.80=almost perfect")

Quadratic Weighted Cohen's Kappa (for model vs human annotator agreement checking)

threat_score QWK= 0.7597
urgency_score QWK= 0.4408
justification_score QWK= 0.4255

And macro-avg QWK= 0.5420

The Landis & Koch benchmarks are:
<0.20=slight, 0.21-0.40=fair, 0.41-0.60=moderate, 0.61-0.80=substantial, >0.80=almost perfect


In [25]:
# Finally I save the test predictions & labels for cross-model comparison against the baseline
# format is (N, 3) float32 numpy. The cross-model comparison scripts load these for paired t-tests against the baseline
np.save("threat_only_aux_test_preds.npy", test_preds.astype("float32"))
np.save("threat_only_aux_test_labels.npy", test_labels.astype("float32"))
print(f"saved threat_only_aux_test_preds.npy. Its shape is {test_preds.shape}")
print(f"saved threat_only_aux_test_labels.npy. Its shape is {test_labels.shape}")

saved threat_only_aux_test_preds.npy. Its shape is (225, 3)
saved threat_only_aux_test_labels.npy. Its shape is (225, 3)


In [26]:
# & the best-checkpoint weights are saved for downstream inference on the full unlabelled corpus (though I only use the weights of the best reported model (see my paper for details))
checkpoint_path= "final_securitisation_classifier_threat_only_aux.pt"
torch.save(model.state_dict(), checkpoint_path)
print(f"all done, model saved in this same directory as '{checkpoint_path}'")

all done, model saved in this same directory as 'final_securitisation_classifier_threat_only_aux.pt'


## References

[1] K. Sechidis, G. Tsoumakas, and I. Vlahavas, "On the stratification of multi-label data," in *Proc. 22nd European Conf. on Machine Learning and Knowledge Discovery in Databases (ECML/PKDD)*, 2011, pp. 145–158, doi: [10.1007/978-3-642-23808-6_10](https://doi.org/10.1007/978-3-642-23808-6_10).

[2] Y. Yang, K. Zha, Y. Chen, H. Wang, and D. Katabi, "Delving into deep imbalanced regression," in *Proc. 38th Int. Conf. on Machine Learning (ICML)*, ser. Proceedings of Machine Learning Research, vol. 139. PMLR, 2021, pp. 11842–11851. [Online]. Available: https://proceedings.mlr.press/v139/yang21m.html.

[3] N. Reimers and I. Gurevych, "Sentence-BERT: Sentence embeddings using Siamese BERT-networks," in *Proc. 2019 Conf. on Empirical Methods in Natural Language Processing (EMNLP)*, 2019, pp. 3982–3992.

[4] H. Inoue, "Multi-sample dropout for accelerated training and better generalization," *arXiv preprint arXiv:1905.09788*, 2019.

[5] X. Liu, P. He, W. Chen, and J. Gao, "Multi-task deep neural networks for natural language understanding," in *Proc. 57th Annual Meeting of the Association for Computational Linguistics (ACL)*, 2019, pp. 4487–4496, doi: [10.18653/v1/P19-1441](https://doi.org/10.18653/v1/P19-1441). [Online]. Available: [https://aclanthology.org/P19-1441](https://aclanthology.org/P19-1441)

[6] J. L. Ba, J. R. Kiros, and G. E. Hinton, "Layer normalization," *arXiv preprint arXiv:1607.06450*, 2016.

[7] J. Howard and S. Ruder, "Universal language model fine-tuning for text classification," in *Proc. 56th Annual Meeting of the Association for Computational Linguistics (ACL)*, 2018, pp. 328–339.

[8] C. Sun, X. Qiu, Y. Xu, and X. Huang, "How to fine-tune BERT for text classification," in *Proc. 18th China National Conf. on Chinese Computational Linguistics*, ser. Lecture Notes in Computer Science. Springer, 2019, pp. 194–206, doi: [10.1007/978-3-030-32381-3_16](https://doi.org/10.1007/978-3-030-32381-3_16).

[9] J. Devlin, M.-W. Chang, K. Lee, and K. Toutanova, "BERT: Pre-training of deep bidirectional transformers for language understanding," in *Proc. 2019 Conf. of the North American Chapter of the ACL: Human Language Technologies (NAACL-HLT)*, 2019, pp. 4171–4186, doi: [10.18653/v1/N19-1423](https://doi.org/10.18653/v1/N19-1423). [Online]. Available: [https://aclanthology.org/N19-1423](https://aclanthology.org/N19-1423)

[10] R. Caruana, "Multitask learning," *Machine Learning*, vol. 28, no. 1, pp. 41–75, 1997, doi: [10.1023/A:1007379606734](https://doi.org/10.1023/A:1007379606734).

[11] A. Ratner, C. De Sa, S. Wu, D. Selsam, and C. Ré, "Data programming: Creating large training sets, quickly," in *Advances in Neural Information Processing Systems (NeurIPS)*, vol. 29, 2016. [Online]. Available: https://proceedings.neurips.cc/paper_files/paper/2016/file/6709e8d64a5f47269ed5cea9f625f7ab-Paper.pdf

[12] I. Loshchilov and F. Hutter, "SGDR: Stochastic gradient descent with warm restarts," in *Proc. 5th Int. Conf. on Learning Representations (ICLR)*, 2017. [Online]. Available: [https://arxiv.org/abs/1608.03983](https://arxiv.org/abs/1608.03983)

[13] P. Micikevicius, S. Narang, J. Alben, G. Diamos, E. Elsen, D. Garcia, B. Ginsburg, M. Houston, O. Kuchaiev, G. Venkatesh, and H. Wu, "Mixed precision training," in *Proc. 6th Int. Conf. on Learning Representations (ICLR)*, 2018.

[14] X. Liang, L. Wu, J. Li, Y. Wang, Q. Meng, T. Qin, W. Chen, M. Zhang, and T.-Y. Liu, "R-Drop: Regularized dropout for neural networks," in *Advances in Neural Information Processing Systems (NeurIPS)*, vol. 34, 2021, pp. 10890–10905. [Online]. Available: https://proceedings.neurips.cc/paper/2021/hash/5a66b9200f29ac3fa0ae244cc2a51b39-Abstract.html

[15] R. Pascanu, T. Mikolov, and Y. Bengio, "On the difficulty of training recurrent neural networks," in *Proc. 30th Int. Conf. on Machine Learning (ICML)*, 2013, pp. 1310–1318. [Online]. Available: [https://arxiv.org/abs/1211.5063](https://arxiv.org/abs/1211.5063)

[16] A. Wang, A. Singh, J. Michael, F. Hill, O. Levy, and S. R. Bowman, "GLUE: A multi-task benchmark and analysis platform for natural language understanding," in *Proc. EMNLP Workshop BlackboxNLP*, 2018, pp. 353–355.

[17] J. Cohen, "Weighted kappa: Nominal scale agreement provision for scaled disagreement or partial credit," *Psychological Bulletin*, vol. 70, no. 4, pp. 213–220, 1968, doi: [10.1037/h0026256](https://doi.org/10.1037/h0026256). [Online]. Available: [https://psycnet.apa.org/doiLanding?doi=10.1037%2Fh0026256](https://psycnet.apa.org/doiLanding?doi=10.1037%2Fh0026256)

[18] J. R. Landis and G. G. Koch, "The measurement of observer agreement for categorical data," *Biometrics*, vol. 33, no. 1, pp. 159–174, 1977, doi: [10.2307/2529310](https://doi.org/10.2307/2529310). [Online]. Available: [https://www.jstor.org/stable/2529310](https://www.jstor.org/stable/2529310)
